# Day 19 — Systematic Validation-Guided Tuning
**Project:** Underwater Garbage Detection  
**Model family:** YOLOv8n  
**Purpose:** change meaningful training settings systematically, record every experiment, select the best configuration using **validation only**, and keep the final test set untouched.

This notebook reuses the existing Dataset V1 and saved Day 17 checkpoint. It does not rebuild or retrain Day 15/16/17 from scratch.

In [12]:
from pathlib import Path
import os, json, time, random, shutil, zipfile, hashlib
from collections import defaultdict
import pandas as pd
import numpy as np

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    pass

# ============================================================
# PROJECT + DATASET PATHS
# These are the same paths used by the completed Day 18 notebook.
# ============================================================
PROJECT_DIR = Path("/content/drive/MyDrive/INTERNSHIPWORK/yolo_runs")
V1_ROOT = Path("/content/TrashCAN_YOLO_Dataset_V1")
DATA_YAML = V1_ROOT / "data.yaml"
SOURCE_ZIP = Path("/content/drive/MyDrive/INTERNSHIPWORK/dataset.zip")
SOURCE_ROOT = Path("/content/TrashCAN")

assert PROJECT_DIR.exists(), f"Missing Drive project folder: {PROJECT_DIR}"

def ensure_dataset_v1():
    """
    Re-create Dataset V1 only when the current Colab runtime does not
    contain it. The construction follows the Dataset V1 procedure used
    in the Day 18 notebook:
      - original material_version train + val
      - exact 8 garbage classes
      - SHA-256 grouping
      - 10% test, then ~10% validation from the remaining pool
      - COCO -> YOLO labels
      - train/val/test + data.yaml
    """
    if DATA_YAML.exists():
        print("Dataset V1 already exists:", DATA_YAML)
        return

    print("\nDataset V1 is missing in this Colab runtime.")
    print("Rebuilding it from the saved Day 16 source dataset...")

    if not SOURCE_ROOT.exists():
        if not SOURCE_ZIP.exists():
            raise FileNotFoundError(
                f"Neither Dataset V1 nor the source ZIP was found.\n"
                f"Expected ZIP: {SOURCE_ZIP}\n"
                "Make sure Google Drive is mounted and dataset.zip is present "
                "under INTERNSHIPWORK."
            )
        print("Extracting:", SOURCE_ZIP)
        SOURCE_ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(SOURCE_ZIP, "r") as z:
            z.extractall(SOURCE_ROOT)
        print("Source dataset extracted.")

    material_path = SOURCE_ROOT / "dataset" / "material_version"
    if not material_path.exists():
        matches = list(SOURCE_ROOT.rglob("material_version"))
        if matches:
            material_path = matches[0]

    if not material_path.exists():
        raise FileNotFoundError(
            f"material_version was not found inside {SOURCE_ROOT}"
        )

    train_path = material_path / "train"
    val_path = material_path / "val"
    train_json_path = material_path / "instances_train_trashcan.json"
    val_json_path = material_path / "instances_val_trashcan.json"

    required = [train_path, val_path, train_json_path, val_json_path]
    for p in required:
        if not p.exists():
            raise FileNotFoundError(f"Required source item not found: {p}")

    with open(train_json_path, "r", encoding="utf-8") as f:
        train_data = json.load(f)
    with open(val_json_path, "r", encoding="utf-8") as f:
        val_data = json.load(f)

    categories = train_data["categories"]
    category_names = {c["id"]: c["name"] for c in categories}

    garbage_classes = {
        "trash_etc", "trash_fabric", "trash_fishing_gear",
        "trash_metal", "trash_paper", "trash_plastic",
        "trash_rubber", "trash_wood"
    }
    garbage_ids = {
        c["id"] for c in categories if c["name"] in garbage_classes
    }

    image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

    def list_images(folder):
        return sorted(
            f for f in os.listdir(folder)
            if f.lower().endswith(image_extensions)
        )

    annotation_lookup = defaultdict(list)
    for ann in train_data["annotations"]:
        annotation_lookup[("train", ann["image_id"])].append(ann)
    for ann in val_data["annotations"]:
        annotation_lookup[("val", ann["image_id"])].append(ann)

    image_records = []
    for source_pool, data, image_folder in [
        ("train", train_data, train_path),
        ("val", val_data, val_path),
    ]:
        for img in data["images"]:
            image_path = image_folder / img["file_name"]
            if not image_path.exists():
                continue

            sha256 = hashlib.sha256()
            with open(image_path, "rb") as f:
                for chunk in iter(lambda: f.read(1024 * 1024), b""):
                    sha256.update(chunk)

            image_records.append({
                "source_pool": source_pool,
                "image_id": int(img["id"]),
                "filename": img["file_name"],
                "image_path": str(image_path),
                "width": int(img["width"]),
                "height": int(img["height"]),
                "sha256": sha256.hexdigest(),
            })

    records_df = pd.DataFrame(image_records)
    print("Indexed images:", len(records_df))

    # Same group-aware split procedure as the Day 18 Dataset V1 builder.
    from sklearn.model_selection import GroupShuffleSplit

    gss_test = GroupShuffleSplit(
        n_splits=1, test_size=0.10, random_state=42
    )
    trainval_idx, test_idx = next(
        gss_test.split(records_df, groups=records_df["sha256"])
    )
    trainval_df = records_df.iloc[trainval_idx].copy()
    test_df = records_df.iloc[test_idx].copy()

    gss_val = GroupShuffleSplit(
        n_splits=1, test_size=1 / 9, random_state=42
    )
    train_idx, val_idx = next(
        gss_val.split(trainval_df, groups=trainval_df["sha256"])
    )
    train_df = trainval_df.iloc[train_idx].copy()
    val_df = trainval_df.iloc[val_idx].copy()

    train_df["split"] = "train"
    val_df["split"] = "val"
    test_df["split"] = "test"
    split_df = pd.concat(
        [train_df, val_df, test_df], ignore_index=True
    )

    cross_split = (
        split_df.groupby("sha256")["split"].nunique()
    )
    cross_split = cross_split[cross_split > 1]
    assert len(cross_split) == 0, "Duplicate image content crossed dataset splits."

    yolo_names = sorted(garbage_classes)
    yolo_class_map = {
        cid: yolo_names.index(category_names[cid])
        for cid in garbage_ids
    }

    def coco_bbox_to_yolo(bbox, image_width, image_height):
        x, y, w, h = bbox
        x1 = max(0, x)
        y1 = max(0, y)
        x2 = min(image_width, x + w)
        y2 = min(image_height, y + h)
        cw, ch = x2 - x1, y2 - y1
        if cw <= 0 or ch <= 0:
            return None
        vals = [
            ((x1 + x2) / 2.0) / image_width,
            ((y1 + y2) / 2.0) / image_height,
            cw / image_width,
            ch / image_height,
        ]
        return [min(1.0, max(0.0, v)) for v in vals]

    if V1_ROOT.exists():
        shutil.rmtree(V1_ROOT)

    for split in ["train", "val", "test"]:
        (V1_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
        (V1_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

    from shutil import copy2
    from tqdm.auto import tqdm

    for _, row in tqdm(
        split_df.iterrows(),
        total=len(split_df),
        desc="Creating Dataset V1"
    ):
        split = row["split"]
        src_pool = row["source_pool"]
        image_id = int(row["image_id"])
        filename = row["filename"]

        dst_img = V1_ROOT / "images" / split / filename
        dst_lbl = V1_ROOT / "labels" / split / f"{Path(filename).stem}.txt"
        copy2(row["image_path"], dst_img)

        yolo_lines = []
        for ann in annotation_lookup[(src_pool, image_id)]:
            cid = int(ann["category_id"])
            if cid not in yolo_class_map:
                continue
            box = coco_bbox_to_yolo(
                ann["bbox"], int(row["width"]), int(row["height"])
            )
            if box is None:
                continue
            yolo_lines.append(
                str(yolo_class_map[cid]) + " " +
                " ".join(f"{v:.6f}" for v in box)
            )

        dst_lbl.write_text("\n".join(yolo_lines), encoding="utf-8")

    try:
        import yaml
    except ImportError:
        raise ImportError("PyYAML is required. Run: !pip install -q pyyaml")

    data_yaml = {
        "path": str(V1_ROOT),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": yolo_names,
    }
    with open(DATA_YAML, "w", encoding="utf-8") as f:
        yaml.safe_dump(data_yaml, f, sort_keys=False, allow_unicode=True)

    split_df.sort_values(["split", "filename"]).to_csv(
        V1_ROOT / "split_manifest.csv", index=False
    )

    print("\nDataset V1 created successfully.")

ensure_dataset_v1()

print("\nPROJECT_DIR:", PROJECT_DIR)
print("DATA_YAML  :", DATA_YAML)
print("TRAIN      :", V1_ROOT / "images" / "train")
print("VAL        :", V1_ROOT / "images" / "val")
print("TEST       :", V1_ROOT / "images" / "test")
print("\nIMPORTANT: Day 19/20 selection uses validation only;")
print("the test split is reserved for Day 21 final evaluation.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Dataset V1 is missing in this Colab runtime.
Rebuilding it from the saved Day 16 source dataset...
Extracting: /content/drive/MyDrive/INTERNSHIPWORK/dataset.zip
Source dataset extracted.
Indexed images: 7212


Creating Dataset V1:   0%|          | 0/7212 [00:00<?, ?it/s]


Dataset V1 created successfully.

PROJECT_DIR: /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs
DATA_YAML  : /content/TrashCAN_YOLO_Dataset_V1/data.yaml
TRAIN      : /content/TrashCAN_YOLO_Dataset_V1/images/train
VAL        : /content/TrashCAN_YOLO_Dataset_V1/images/val
TEST       : /content/TrashCAN_YOLO_Dataset_V1/images/test

IMPORTANT: Day 19/20 selection uses validation only;
the test split is reserved for Day 21 final evaluation.


In [13]:
# Install/use the same Ultralytics family used in the supplied Day 17 notebook.
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.155"])

from ultralytics import YOLO
import ultralytics
print("Ultralytics:", ultralytics.__version__)

Ultralytics: 8.4.155


## 1. Automatically locate the completed Day 17 checkpoint

The code prefers a completed `day17_yolov8n_*` run with `results.csv` and `weights/best.pt`, and selects the run with the greatest completed epoch count. If you know the exact intended run, set `DAY17_RUN_OVERRIDE`.

In [14]:
DAY17_RUN_OVERRIDE = None  # Example: PROJECT_DIR / "day17_yolov8n_tuned_continuation"

def completed_runs(prefix):
    rows = []
    for run_dir in sorted(PROJECT_DIR.glob(prefix + "*")):
        csv_file = run_dir / "results.csv"
        best_pt = run_dir / "weights" / "best.pt"
        if not csv_file.exists() or not best_pt.exists():
            continue
        try:
            df = pd.read_csv(csv_file)
            df.columns = [c.strip() for c in df.columns]
            if "epoch" in df.columns and len(df):
                rows.append({
                    "run": run_dir.name,
                    "run_dir": run_dir,
                    "epochs": int(df["epoch"].max()) + 1,
                    "csv": csv_file,
                    "best_pt": best_pt
                })
        except Exception as e:
            print("Skipping", run_dir.name, ":", e)
    return sorted(rows, key=lambda x: x["epochs"], reverse=True)

if DAY17_RUN_OVERRIDE is not None:
    DAY17_RUN_DIR = Path(DAY17_RUN_OVERRIDE)
else:
    candidates = completed_runs("day17_yolov8n_")
    if not candidates:
        raise FileNotFoundError("No completed Day 17 run with results.csv + best.pt was found.")
    DAY17_RUN_DIR = candidates[0]["run_dir"]

DAY17_BEST = DAY17_RUN_DIR / "weights" / "best.pt"
DAY17_CSV = DAY17_RUN_DIR / "results.csv"

print("Selected Day 17 run:", DAY17_RUN_DIR)
print("Checkpoint:", DAY17_BEST)
assert DAY17_BEST.exists()

Selected Day 17 run: /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day17_yolov8n_tuned_continuation
Checkpoint: /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day17_yolov8n_tuned_continuation/weights/best.pt


In [15]:
# Read the Day 17 validation history. This is validation data only.
r17 = pd.read_csv(DAY17_CSV)
r17.columns = [c.strip() for c in r17.columns]

required = ["epoch", "metrics/mAP50(B)", "metrics/mAP50-95(B)",
            "metrics/precision(B)", "metrics/recall(B)"]
missing = [c for c in required if c not in r17.columns]
if missing:
    raise KeyError(f"Missing expected Ultralytics columns: {missing}")

best17 = r17.loc[r17["metrics/mAP50-95(B)"].idxmax()]
p = float(best17["metrics/precision(B)"]); r = float(best17["metrics/recall(B)"])
f1 = 2*p*r/(p+r) if p+r else 0.0

print("Day 17 best validation epoch:", int(best17["epoch"]))
print("mAP50:", round(float(best17["metrics/mAP50(B)"]), 5))
print("mAP50-95:", round(float(best17["metrics/mAP50-95(B)"]), 5))
print("Precision:", round(p, 5), "Recall:", round(r, 5), "F1:", round(f1, 5))

Day 17 best validation epoch: 86
mAP50: 0.57704
mAP50-95: 0.39734
Precision: 0.77196 Recall: 0.5407 F1: 0.63596


## 2. Controlled experiment plan

Only one meaningful group of settings is changed per experiment.

- **E0 — Day 17 checkpoint control:** no new training; reference validation result.
- **E1 — Visibility-focused augmentation:** stronger but still bounded brightness/contrast and HSV variation.
- **E2 — Small-object-focused resolution:** larger training image size with conservative scale/mosaic.
- **E3 — Learning-rate refinement:** lower initial LR and weight decay while keeping the E1/E2 augmentation family conservative.

All experiments use the same training split and the same validation split. The test split is not read during this selection stage.

In [16]:
EXPERIMENTS = {
    "E1_visibility_aug": {
        "imgsz": 832,
        "epochs": 15,
        "batch": 8,
        "lr0": 0.0005,
        "lrf": 0.01,
        "weight_decay": 0.0005,
        "hsv_h": 0.015,
        "hsv_s": 0.55,
        "hsv_v": 0.40,
        "degrees": 0.0,
        "translate": 0.10,
        "scale": 0.25,
        "fliplr": 0.50,
        "mosaic": 0.70,
        "mixup": 0.0,
    },
    "E2_small_object": {
        "imgsz": 960,
        "epochs": 15,
        "batch": 8,
        "lr0": 0.0005,
        "lrf": 0.01,
        "weight_decay": 0.0005,
        "hsv_h": 0.015,
        "hsv_s": 0.40,
        "hsv_v": 0.30,
        "degrees": 0.0,
        "translate": 0.08,
        "scale": 0.20,
        "fliplr": 0.50,
        "mosaic": 0.50,
        "mixup": 0.0,
    },
    "E3_lr_refinement": {
        "imgsz": 832,
        "epochs": 15,
        "batch": 8,
        "lr0": 0.00025,
        "lrf": 0.005,
        "weight_decay": 0.00075,
        "hsv_h": 0.015,
        "hsv_s": 0.40,
        "hsv_v": 0.30,
        "degrees": 0.0,
        "translate": 0.08,
        "scale": 0.20,
        "fliplr": 0.50,
        "mosaic": 0.50,
        "mixup": 0.0,
    },
}

TUNING_ROOT = PROJECT_DIR / "day19_systematic_tuning"
TUNING_ROOT.mkdir(parents=True, exist_ok=True)

# Set True when you are ready to run the experiments.
RUN_NEW_EXPERIMENTS = True
print("Experiments:", list(EXPERIMENTS))
print("Output root:", TUNING_ROOT)

Experiments: ['E1_visibility_aug', 'E2_small_object', 'E3_lr_refinement']
Output root: /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day19_systematic_tuning


In [17]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU count: 1
GPU: Tesla T4


In [18]:
import torch

assert torch.cuda.is_available(), (
    "GPU is not available. Go to Runtime → Change runtime type → T4 GPU."
)

print("GPU ready:", torch.cuda.get_device_name(0))

GPU ready: Tesla T4


In [19]:
DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("Selected device:", DEVICE)

if DEVICE == "cpu":
    raise RuntimeError(
        "CUDA GPU is not available. "
        "Please enable a T4 GPU before running Day 19 training."
    )

Selected device: 0


In [20]:
!pip install -q ultralytics==8.4.155

In [21]:
# ============================================================
# DAY 19 — RUN CONTROLLED EXPERIMENTS
# ============================================================

import time
import torch
from ultralytics import YOLO

# ------------------------------------------------------------
# 1. Control whether new experiments should run
# ------------------------------------------------------------
RUN_NEW_EXPERIMENTS = True

# ------------------------------------------------------------
# 2. Check GPU before starting
# ------------------------------------------------------------
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is not available.\n"
        "Please enable T4 GPU in Colab:\n"
        "Runtime → Change runtime type → T4 GPU\n"
        "Then restart the runtime and run the setup cells again."
    )

DEVICE = 0

print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("Using device:", DEVICE)

# ------------------------------------------------------------
# 3. Check required variables
# ------------------------------------------------------------
required_variables = {
    "EXPERIMENTS": EXPERIMENTS,
    "TUNING_ROOT": TUNING_ROOT,
    "DAY17_BEST": DAY17_BEST,
    "DATA_YAML": DATA_YAML,
}

print("\nChecking required paths...")

for name, value in required_variables.items():
    print(f"{name}: {value}")

# ------------------------------------------------------------
# 4. Verify important files
# ------------------------------------------------------------
assert DATA_YAML.exists(), (
    f"Dataset YAML not found:\n{DATA_YAML}"
)

assert DAY17_BEST.exists(), (
    f"Day 17 best checkpoint not found:\n{DAY17_BEST}"
)

TUNING_ROOT.mkdir(parents=True, exist_ok=True)

print("\nAll required files are available.")
print("Day 17 checkpoint:", DAY17_BEST)
print("Dataset YAML:", DATA_YAML)
print("Experiment root:", TUNING_ROOT)

# ------------------------------------------------------------
# 5. Run experiments
# ------------------------------------------------------------
experiment_records = []

if RUN_NEW_EXPERIMENTS:

    for exp_name, cfg in EXPERIMENTS.items():

        run_name = f"day19_{exp_name}"
        run_dir = TUNING_ROOT / run_name

        csv_path = run_dir / "results.csv"
        best_path = run_dir / "weights" / "best.pt"

        # ----------------------------------------------------
        # Skip already completed experiment
        # ----------------------------------------------------
        if csv_path.exists() and best_path.exists():

            print("\n" + "-" * 80)
            print(f"SKIPPING COMPLETED EXPERIMENT: {run_name}")
            print("-" * 80)

            experiment_records.append({
                "experiment": exp_name,
                "status": "already_completed",
                "hours": 0.0,
                "run_dir": str(run_dir)
            })

            continue

        # ----------------------------------------------------
        # Run experiment
        # ----------------------------------------------------
        print("\n" + "=" * 80)
        print("RUNNING:", run_name)
        print("=" * 80)

        print("Configuration:")
        for key, value in cfg.items():
            print(f"  {key}: {value}")

        print("\nStarting from Day 17 best:")
        print(DAY17_BEST)

        model = YOLO(str(DAY17_BEST))

        start = time.time()

        model.train(
            data=str(DATA_YAML),
            project=str(TUNING_ROOT),
            name=run_name,

            # IMPORTANT:
            # We are loading the Day 17 model above.
            pretrained=False,

            # GPU
            device=DEVICE,

            # Training control
            patience=8,

            # Save/plots
            plots=True,
            save=True,
            verbose=True,

            # Experiment-specific settings
            **cfg
        )

        elapsed = time.time() - start

        experiment_records.append({
            "experiment": exp_name,
            "status": "completed",
            "hours": elapsed / 3600,
            "run_dir": str(run_dir)
        })

        print("\nCompleted:", run_name)
        print(f"Time: {elapsed / 3600:.2f} hours")

else:

    print(
        "\nRUN_NEW_EXPERIMENTS = False\n"
        "No new training experiments were started."
    )

# ------------------------------------------------------------
# 6. Final experiment summary
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("DAY 19 EXPERIMENT SUMMARY")
print("=" * 80)

for record in experiment_records:
    print(record)

GPU available: True
GPU: Tesla T4
Using device: 0

Checking required paths...
EXPERIMENTS: {'E1_visibility_aug': {'imgsz': 832, 'epochs': 15, 'batch': 8, 'lr0': 0.0005, 'lrf': 0.01, 'weight_decay': 0.0005, 'hsv_h': 0.015, 'hsv_s': 0.55, 'hsv_v': 0.4, 'degrees': 0.0, 'translate': 0.1, 'scale': 0.25, 'fliplr': 0.5, 'mosaic': 0.7, 'mixup': 0.0}, 'E2_small_object': {'imgsz': 960, 'epochs': 15, 'batch': 8, 'lr0': 0.0005, 'lrf': 0.01, 'weight_decay': 0.0005, 'hsv_h': 0.015, 'hsv_s': 0.4, 'hsv_v': 0.3, 'degrees': 0.0, 'translate': 0.08, 'scale': 0.2, 'fliplr': 0.5, 'mosaic': 0.5, 'mixup': 0.0}, 'E3_lr_refinement': {'imgsz': 832, 'epochs': 15, 'batch': 8, 'lr0': 0.00025, 'lrf': 0.005, 'weight_decay': 0.00075, 'hsv_h': 0.015, 'hsv_s': 0.4, 'hsv_v': 0.3, 'degrees': 0.0, 'translate': 0.08, 'scale': 0.2, 'fliplr': 0.5, 'mosaic': 0.5, 'mixup': 0.0}}
TUNING_ROOT: /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day19_systematic_tuning
DAY17_BEST: /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day17_

In [22]:
# Build the Day 19 experiment log from the ACTUAL saved results.csv files.
def validation_summary(run_dir, label):
    csv_path = run_dir / "results.csv"
    best_pt = run_dir / "weights" / "best.pt"
    if not csv_path.exists() or not best_pt.exists():
        return None
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    row = df.loc[df["metrics/mAP50-95(B)"].idxmax()]
    p = float(row["metrics/precision(B)"]); r = float(row["metrics/recall(B)"])
    return {
        "experiment": label,
        "run_dir": str(run_dir),
        "best_epoch": int(row["epoch"]),
        "mAP50": float(row["metrics/mAP50(B)"]),
        "mAP50-95": float(row["metrics/mAP50-95(B)"]),
        "precision": p,
        "recall": r,
        "F1": 2*p*r/(p+r) if p+r else 0.0,
        "best_pt": str(best_pt),
    }

rows = [{
    "experiment": "E0_day17_control",
    "run_dir": str(DAY17_RUN_DIR),
    "best_epoch": int(best17["epoch"]),
    "mAP50": float(best17["metrics/mAP50(B)"]),
    "mAP50-95": float(best17["metrics/mAP50-95(B)"]),
    "precision": float(best17["metrics/precision(B)"]),
    "recall": float(best17["metrics/recall(B)"]),
    "F1": f1,
    "best_pt": str(DAY17_BEST),
}]

for exp_name in EXPERIMENTS:
    item = validation_summary(TUNING_ROOT / f"day19_{exp_name}", exp_name)
    if item:
        rows.append(item)

day19_table = pd.DataFrame(rows).sort_values("mAP50-95", ascending=False).reset_index(drop=True)
display(day19_table)

log = {
    "day": 19,
    "selection_metric": "validation mAP50-95(B)",
    "test_used_for_selection": False,
    "dataset_yaml": str(DATA_YAML),
    "day17_starting_checkpoint": str(DAY17_BEST),
    "experiments": EXPERIMENTS,
    "results": day19_table.to_dict(orient="records")
}
with open(TUNING_ROOT / "day19_experiment_log.json", "w", encoding="utf-8") as f:
    json.dump(log, f, indent=2)

day19_table.to_csv(TUNING_ROOT / "day19_experiment_log.csv", index=False)
print("Saved:", TUNING_ROOT / "day19_experiment_log.json")

,experiment,run_dir,best_epoch,mAP50,mAP50-95,precision,recall,F1,best_pt
0,E0_day17_control,/content/drive/MyDrive/INTERNSHIPWORK/yolo_run...,86,0.57704,0.39734,0.77196,0.54070,0.635959,/content/drive/MyDrive/INTERNSHIPWORK/yolo_run...
1,E3_lr_refinement,/content/drive/MyDrive/INTERNSHIPWORK/yolo_run...,15,0.53232,0.36077,0.59789,0.49376,0.540859,/content/drive/MyDrive/INTERNSHIPWORK/yolo_run...
2,E2_small_object,/content/drive/MyDrive/INTERNSHIPWORK/yolo_run...,15,0.47145,0.32022,0.62833,0.42053,0.503845,/content/drive/MyDrive/INTERNSHIPWORK/yolo_run...
3,E1_visibility_aug,/content/drive/MyDrive/INTERNSHIPWORK/yolo_run...,15,0.44723,0.30297,0.42357,0.44899,0.435910,/content/drive/MyDrive/INTERNSHIPWORK/yolo_run...


Saved: /content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day19_systematic_tuning/day19_experiment_log.json


In [23]:
# Select and SAVE the best validated configuration.
# No test-set metric is used here.
best_row = day19_table.iloc[0]
BEST_DAY19_RUN = Path(best_row["run_dir"])
BEST_DAY19_PT = Path(best_row["best_pt"])

best_config = {
    "selected_by": "validation mAP50-95(B)",
    "test_used": False,
    "experiment": str(best_row["experiment"]),
    "run_dir": str(BEST_DAY19_RUN),
    "best_pt": str(BEST_DAY19_PT),
    "validation_metrics": {
        "mAP50": float(best_row["mAP50"]),
        "mAP50-95": float(best_row["mAP50-95"]),
        "precision": float(best_row["precision"]),
        "recall": float(best_row["recall"]),
        "F1": float(best_row["F1"]),
    },
    "dataset_yaml": str(DATA_YAML),
}

with open(TUNING_ROOT / "BEST_VALIDATED_CONFIG.json", "w", encoding="utf-8") as f:
    json.dump(best_config, f, indent=2)

print(json.dumps(best_config, indent=2))

{
  "selected_by": "validation mAP50-95(B)",
  "test_used": false,
  "experiment": "E0_day17_control",
  "run_dir": "/content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day17_yolov8n_tuned_continuation",
  "best_pt": "/content/drive/MyDrive/INTERNSHIPWORK/yolo_runs/day17_yolov8n_tuned_continuation/weights/best.pt",
  "validation_metrics": {
    "mAP50": 0.57704,
    "mAP50-95": 0.39734,
    "precision": 0.77196,
    "recall": 0.5407,
    "F1": 0.635958697606387
  },
  "dataset_yaml": "/content/TrashCAN_YOLO_Dataset_V1/data.yaml"
}


### Day 19 output

The notebook creates:

- `day19_systematic_tuning/day19_experiment_log.csv`
- `day19_systematic_tuning/day19_experiment_log.json`
- `day19_systematic_tuning/BEST_VALIDATED_CONFIG.json`
- one saved run folder per experiment containing `results.csv`, plots and `weights/best.pt`

The selected configuration is selected **only from validation performance**.